# CoDA Continuous: Trace Evolution
This notebook visualizes the evolution of the accumulated eligibility traces (Counts) for Reward (`C[0]`) and Non-Reward (`C[1]`) outcomes over the course of training.
We place the Cue in the center of the environment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from coda_continuous import CuedContinuousBox, TerminalRBFEncoder, LinearDeltaModel, CoDAContingencyContinuous

# Setup environment with Cue in Center
def make_env_center_cue(seed=42):
    env = CuedContinuousBox(seed=seed, goal_radius=0.2,cue_radius=0.15, step_size=1)
    # env.CUE = np.array([0.3, 0.3], dtype=np.float32)  # lower left
    env.CUE = np.array([0.5, 0.5], dtype=np.float32)  # center
    return env

env = make_env_center_cue()
print(f"Cue location: {env.CUE}")
print(f"Goal location: {env.GOAL}")

In [ ]:
# Helper to visualize the stored "Concept" (Accumulated Trace) C[k]
def get_response_map(agent, fe, outcome_k, grid_n=50):
    xs = np.linspace(0, 1, grid_n)
    ys = np.linspace(0, 1, grid_n)
    XX, YY = np.meshgrid(xs, ys)
    
    Z = np.zeros_like(XX)
    
    # We want to visualize C[outcome_k] @ phi(x,y)
    # Be careful: C is (K, M). C[k] is (M,)
    # It's faster to precompute phi for all grid points if M is small, 
    # but here we can just loop or batch.
    
    Ck_weights = agent.C[outcome_k]
    
    # Optimization: compute phi for all centers at once?
    # RBF output is sum of weights * gaussian(center)
    # It is effectively constructing a function f(x) = sum w_i * k(x_i, x)
    
    # Let's just do a naive loop for clarity and to reuse fe.phi
    for iy in range(grid_n):
        for ix in range(grid_n):
            obs_x, obs_y = XX[iy, ix], YY[iy, ix]
            # Use non-terminal phi for visualization of the "field"
            phi = fe.phi(obs_x, obs_y, terminal=False)
            Z[iy, ix] = np.dot(Ck_weights, phi)
            
    return XX, YY, Z

def plot_maps(agent, fe, trial_idx, threshold=0.7):
    XX, YY, Z_rew = get_response_map(agent, fe, outcome_k=0) # Reward
    XX, YY, Z_non = get_response_map(agent, fe, outcome_k=1) # Non-Reward
    
    # Calculate Contingency Map: E_r / (E_r + E_nr)
    denominator = Z_rew + Z_non
    # Avoid division by zero: if denom is practically 0, set ratio to 0.5 (neutral)
    Z_cont = np.divide(Z_rew, denominator, out=np.full_like(Z_rew, 0.5), where=denominator > 1e-9)

    # Mask: Locate areas > threshold
    Z_mask = (Z_cont > threshold).astype(float)

    # Find max contingency location
    max_idx = np.unravel_index(np.argmax(Z_cont, axis=None), Z_cont.shape)
    max_y_idx, max_x_idx = max_idx
    max_x = XX[max_y_idx, max_x_idx]
    max_y = YY[max_y_idx, max_x_idx]
    max_val = Z_cont[max_y_idx, max_x_idx]
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    
    vmin = min(Z_rew.min(), Z_non.min())
    vmax = max(Z_rew.max(), Z_non.max())
    
    # 1. Reward Trace
    ax = axes[0]
    im = ax.imshow(Z_rew, origin="lower", extent=[0,1,0,1], vmin=vmin, vmax=vmax)
    ax.set_title(f"Reward Trace (Trial {trial_idx})")
    ax.scatter([0.5], [0.5], c='white', marker='x', label='Cue')
    ax.scatter([env.GOAL[0]], [env.GOAL[1]], c='yellow', marker='*', label='Goal')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    # 2. Non-Reward Trace
    ax = axes[1]
    im = ax.imshow(Z_non, origin="lower", extent=[0,1,0,1], vmin=vmin, vmax=vmax)
    ax.set_title(f"Non-Reward Trace (Trial {trial_idx})")
    ax.scatter([0.5], [0.5], c='white', marker='x')
    ax.scatter([env.GOAL[0]], [env.GOAL[1]], c='yellow', marker='*')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # 3. Contingency Map
    ax = axes[2]
    im = ax.imshow(Z_cont, origin="lower", extent=[0,1,0,1], vmin=0.0, vmax=1.0, cmap='RdYlBu_r')
    ax.set_title(f"Contingency E_r/(E_r+E_nr)")
    ax.scatter([0.5], [0.5], c='white', marker='x')
    ax.scatter([env.GOAL[0]], [env.GOAL[1]], c='yellow', marker='*')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # 4. Masked Map
    ax = axes[3]
    im = ax.imshow(Z_mask, origin="lower", extent=[0,1,0,1], vmin=0.0, vmax=1.0, cmap='gray')
    ax.set_title(f"Mask: Contingency > {threshold}")
    ax.scatter([0.5], [0.5], c='red', marker='x')
    ax.scatter([env.GOAL[0]], [env.GOAL[1]], c='yellow', marker='*')
    # Plot Max
    ax.scatter([max_x], [max_y], c='lime', marker='o', s=100, edgecolors='black', label=f'Max ({max_val:.2f})')
    ax.legend(loc='upper left', fontsize='small')

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Training Loop identifying checkpoints
seed = 42
n_episodes = 1000
checkpoints = range(0, n_episodes, 50) # Plot every 50 episodes
step_size = 0.15

env = make_env_center_cue(seed)
fe = TerminalRBFEncoder(n_per_dim=8, sigma_normal=0.15, sigma_nocue=0.08, noncue_factor=0.001)
trans_model = LinearDeltaModel() # Not strictly used for eligibility but part of init
agent = CoDAContingencyContinuous(n_outcomes=2, feat_dim=fe.M, transition_model=trans_model, 
                                  gamma=0.99, lam=0.95)

rng = np.random.default_rng(seed+1)

def plot_trajectory(env, trajectory, outcome, trial_idx):
    trait_x = [p[0] for p in trajectory]
    trait_y = [p[1] for p in trajectory]
    
    plt.figure(figsize=(5, 5))
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    
    # Plot Environment
    # Goal
    goal_circle = plt.Circle(env.GOAL, env.goal_radius, color='gold', alpha=0.3, label='Goal')
    plt.gca().add_patch(goal_circle)
    # Cue
    cue_circle = plt.Circle(env.CUE, env.cue_radius, color='blue', alpha=0.3, label='Cue')
    plt.gca().add_patch(cue_circle)
    
    # Plot Trajectory
    plt.plot(trait_x, trait_y, 'k.-', alpha=0.6, label='Path')
    plt.plot(trait_x[0], trait_y[0], 'go', label='Start')
    plt.plot(trait_x[-1], trait_y[-1], 'ro', label='End')
    
    plt.title(f"Episode {trial_idx}, Outcome: {outcome}")
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.grid(True)
    plt.show()

print("Starting Training...")

for ep in range(n_episodes):
    obs = env.reset()
    agent.reset_episode()
    
    trajectory = [obs]
    
    while True:
        x, y = obs
        # Normal features
        phi_xy = fe.phi(x, y, terminal=False)
        agent.step_features(phi_xy)
                # Discrete Action: Randomly choose Right or Down
        # step_size determines magnitude. If step_size >= 1, it moves at max speed of env.
        if rng.random() < 0.5:
             # Right
             a_raw = np.array([step_size, 0.0], dtype=np.float32)
        else:
             # Down
             a_raw = np.array([0.0, -step_size], dtype=np.float32)
        
        # # Random action: x positive (right), y negative (down)
        # dx = rng.uniform(0.0, step_size)
        # dy = rng.uniform(-step_size, 0.0)
        # a_raw = np.array([dx, dy], dtype=np.float32)
        # print(a_raw)



        obs_next, done, outcome_k, info = env.step(a_raw)
        
        trajectory.append(obs_next)
        
        obs = obs_next
        if done:
            xT, yT = obs
            # Terminal features
            phi_T = fe.phi(xT, yT, terminal=True, crossed_cue=info["crossed_cue"])
            agent.step_features(phi_T)
            
            agent.end_episode(outcome_k)
            # print(f"Episode {ep} completed.")
            break
            
    
    if ep in checkpoints:
        plot_trajectory(env, trajectory, outcome_k, ep)
        plot_maps(agent, fe, ep)

print("Training Complete.")